---
# **Lab11: PyTorch and GPU**
---

`torch.cuda` report you can use to understand the CUDA/GPU state of your PyTorch environment

In [8]:
import torch
import sys

def torch_cuda_report():
    print("=== TORCH.CUDA REPORT ===")
    print(f"PyTorch version:       {torch.__version__}")
    print(f"Python version:        {sys.version}")
    print(f"CUDA available:        {torch.cuda.is_available()}")
    print(f"CUDA version (runtime):{torch.version.cuda}")
    print(f"cuDNN version:         {torch.backends.cudnn.version()}")
    print(f"GPU count:             {torch.cuda.device_count()}")
    
    if torch.cuda.is_available():
        print("\n--- DEVICE DETAILS ---")
        for i in range(torch.cuda.device_count()):
            print(f"Device {i}:            {torch.cuda.get_device_name(i)}")
            print(f"  Capability:          {torch.cuda.get_device_capability(i)}")
            print(f"  Device memory:       {torch.cuda.get_device_properties(i).total_memory / (1024**3):.2f} GB")
            print(f"  Current device?      {i == torch.cuda.current_device()}")
            print()

    vram_allocated = torch.cuda.memory_allocated()
    vram_reserved = torch.cuda.memory_reserved()

    print("--- MEMORY STATUS ---")
    print(f"VRAM allocated:        {vram_allocated / (1024**2):.2f} MB")
    print(f"VRAM reserved:         {vram_reserved / (1024**2):.2f} MB")
    print("========================")

torch_cuda_report()

=== TORCH.CUDA REPORT ===
PyTorch version:       2.9.1+cu130
Python version:        3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
CUDA available:        True
CUDA version (runtime):13.0
cuDNN version:         91200
GPU count:             1

--- DEVICE DETAILS ---
Device 0:            NVIDIA GeForce RTX 2060
  Capability:          (7, 5)
  Device memory:       6.00 GB
  Current device?      True

--- MEMORY STATUS ---
VRAM allocated:        9.00 MB
VRAM reserved:         194.00 MB


In [9]:
import torch
from torch.utils.benchmark import Timer

def timer(cmd):
    median = (
        Timer(cmd, globals=globals())
        .adaptive_autorange(min_run_time=1.0, max_run_time=20.0)
        .median
        * 1000
    )
    print(f"{cmd}: {median: 4.4f} ms")
    return median

# A tensor in pageable memory
pageable_tensor = torch.randn(1_000_000)
# A tensor in page-locked (pinned) memory
pinned_tensor = torch.randn(1_000_000, pin_memory=True)
# Runtimes:
pageable_to_device = timer("pageable_tensor.to('cuda:0')")
pinned_to_device = timer("pinned_tensor.to('cuda:0')")
pin_mem = timer("pageable_tensor.pin_memory()")
pin_mem_to_device = timer("pageable_tensor.pin_memory().to('cuda:0')")

pageable_tensor.to('cuda:0'):  0.3645 ms
pinned_tensor.to('cuda:0'):  0.3208 ms
pageable_tensor.pin_memory():  0.0592 ms
pageable_tensor.pin_memory().to('cuda:0'):  0.3849 ms


In [10]:
import torch
import time
device = torch.device("cuda")
# Create CPU tensor and pin it
x_cpu = torch.randn(50000, 30000).pin_memory()
# Start transfer (async)
start = time.time()
x_gpu = x_cpu.to(device, non_blocking=True)
# CPU continues immediately
cpu_work = sum(i for i in range(100_000_000)) # Simulated CPU work
# Ensure transfer is complete before GPU use
torch.cuda.synchronize()
end = time.time()
print("Total elapsed time:", end - start)


Total elapsed time: 2.0613064765930176


# ✅ DataLoader Prefetch + Overlap (Pinned Memory & Streams)

### Goal

Overlap **CPU data loading + H2D transfer** with **GPU compute** by:
- Using `DataLoader(pin_memory=True)` so batches arrive in pinned host memory.
- Using a **dedicated copy stream** to enqueue `to(device, non_blocking=True)`.
- Using `wait_stream` to ensure the default/compute stream waits only when needed

---

In [15]:
## Minimal Prefetch Loader Wrapper


import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim


class CUDAPrefetcher:
    """
    Prefetches the next batch to GPU on a dedicated CUDA stream.
    Assumes DataLoader(pin_memory=True) so H2D can be async.
    """
    def __init__(self, loader, device):
        self.loader = iter(loader)
        self.device = device
        self.stream = torch.cuda.Stream(device=device)
        self.next_batch = None
        self._preload()

    def _to_device(self, batch):
        # Supports (x, y) tuples or arbitrary nested structures if you extend it
        if isinstance(batch, (tuple, list)):
            return [t.to(self.device, non_blocking=True) for t in batch]
        return batch.to(self.device, non_blocking=True)

    def _preload(self):
        try:
            batch = next(self.loader)
        except StopIteration:
            self.next_batch = None
            return

        with torch.cuda.stream(self.stream):
            self.next_batch = self._to_device(batch)

    def __iter__(self):
        return self

    def __next__(self):
        if self.next_batch is None:
            raise StopIteration

        # Wait for the copy stream to finish transferring the batch
        torch.cuda.current_stream(self.device).wait_stream(self.stream)

        batch = self.next_batch
        self._preload()
        return batch


# Training Loop Using Prefetch Overlap
device = torch.device("cuda:0")

# Example dummy dataset/loader
X = torch.randn(10000, 1024)
Y = torch.randint(0, 10, (10000,))
ds = list(zip(X, Y))

loader = DataLoader(
    ds,
    batch_size=256,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

prefetch_loader = CUDAPrefetcher(loader, device)

model = nn.Sequential(
    nn.Linear(1024, 2048), 
    nn.ReLU(), 
    nn.Linear(2048, 10)
).to(device)
opt = optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

model.train()
for xb, yb in prefetch_loader:
    # xb, yb are already on GPU (transferred on copy stream)
    opt.zero_grad()
    logits = model(xb)
    loss = loss_fn(logits, yb)
    loss.backward()
    opt.step()
    


In [12]:
import torch

def h2d_ms(x_cpu, iters=50):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        x_gpu = x_cpu.to("cuda", non_blocking=True)
    end.record()
    end.synchronize()
    return start.elapsed_time(end) / iters

x_cpu = torch.randn(5000, 5000).pin_memory()
h2d_ms(x_cpu)

8.044082641601562

# ✅ CIFAR-10 Classification (Overlap H2D + Compute)

In [17]:
## Complete Example: CIFAR-10 Classification with Stream-Based Prefetch (Overlap H2D + Compute)

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

assert torch.cuda.is_available(), "CUDA GPU required for this example."
device = torch.device("cuda:0")

# -------------------------
# 1) Data: CIFAR-10
# -------------------------
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

data_root = "./data"

train_ds = datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform_train)
test_ds  = datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform_test)

# Important knobs:
# - pin_memory=True enables true async H2D on non_blocking copies
# - persistent_workers=True reduces worker startup overhead
# - num_workers should be tuned for your CPU
train_loader = DataLoader(
    train_ds,
    batch_size=256,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

# -------------------------
# 2) Simple CNN model
# -------------------------
class SmallCIFARCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16x16
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 8x8
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256), nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.net(x)

model = SmallCIFARCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)
    
# -------------------------
# 4) Train / Eval loops
# -------------------------
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    prefetch = CUDAPrefetcher(loader, device)
    
    for inputs, targets in prefetch:
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss = criterion(outputs, targets)
        loss.backward()
        
        optimizer.step()
        
        with torch.no_grad():
            running_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)
            

    return running_loss / max(1, len(loader)), correct / max(1, total)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    # For evaluation, you can still use pinned memory + non_blocking.
    # Prefetching helps less because eval often has less CPU work, but it’s fine.
    prefetch = CUDAPrefetcher(loader, device)

    for inputs, targets in prefetch:
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return running_loss / max(1, len(loader)), correct / max(1, total)

# -------------------------
# 5) Run training
# -------------------------
epochs = 10
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch:02d} | "
          f"train loss {train_loss:.4f} acc {train_acc*100:.2f}% | "
          f"test loss {test_loss:.4f} acc {test_acc*100:.2f}%")


Epoch 01 | train loss 1.7323 acc 36.85% | test loss 1.3938 acc 50.58%
Epoch 02 | train loss 1.4311 acc 48.43% | test loss 1.2286 acc 56.44%
Epoch 03 | train loss 1.2761 acc 54.37% | test loss 1.1295 acc 59.56%
Epoch 04 | train loss 1.1504 acc 58.90% | test loss 1.0165 acc 64.24%
Epoch 05 | train loss 1.0540 acc 62.69% | test loss 0.9145 acc 67.89%
Epoch 06 | train loss 0.9799 acc 65.37% | test loss 0.8720 acc 69.74%
Epoch 07 | train loss 0.9164 acc 67.60% | test loss 0.8178 acc 70.84%
Epoch 08 | train loss 0.8696 acc 69.40% | test loss 0.7624 acc 73.16%
Epoch 09 | train loss 0.8174 acc 71.29% | test loss 0.7445 acc 73.52%
Epoch 10 | train loss 0.7819 acc 72.61% | test loss 0.7262 acc 74.41%


# ✅ Pytorch Profiler

In [ ]:
import torch
import torch.nn as nn

class model_circle(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(2, 10)
        self.layer_2 = nn.Linear(10, 10)
        self.layer_3 = nn.Linear(10, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x)))))

In [ ]:
from torch.profiler import profile, ProfilerActivity, record_function


# define the model 
model = model_circle()
inputs = torch.randn(1024, 2)

with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    with record_function("model_inference"):
        model(inputs)
        
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

**Understanding PyTorch Profiler Output (CPU Table)**

- `aten::linear` → fully connected layer
- `aten::addmm` → matrix multiplication + bias
- `aten::t` → transpose
- `aten::relu` → activation
- `model_inference` → user-defined profiling scope

- **Self CPU %**
    - Percentage of total CPU time spent **inside this operator only**.
    - Does *not* include time spent in child operations.

-** Self CPU**
    - Absolute time spent only inside this operator. Units: microseconds (µs).

- **CPU total %**
    - Percentage of total CPU time including this op and all sub-ops it called.

- **CPU total**
    - Total time including child operations.

- **CPU time avg**
    - Average execution time per call.

- **Num of Calls**
    - How many times this operator was executed.

- **Input Shapes**
    - Shapes of tensors passed to the operator.


**Reconstructing the Model from Shapes**

First `aten::linear`:

```
[[256, 2], [10, 2], [10]]
```

Meaning:

- Batch size: 256
- Input features: 2
- Weight shape: (10, 2)
- Bias shape: (10)

So:

$$
2 \rightarrow 10
$$

---

Second `aten::linear`:

```
[[256, 10], [10, 10], [10]]
```

So:

$$
10 \rightarrow 10
$$

---

Third `aten::linear`:

```
[[256, 10], [1, 10], [1]]
```

So:

$$
10 \rightarrow 1
$$


To see GPU timings:

```python
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA])
```

This profiler output tells us:

- How much CPU time each ATen operator consumed
- Which operators dominate dispatch cost
- The likely structure of the model
- Where optimization efforts may be focused

It does **not** measure raw GPU compute performance.

In [ ]:
device = "cuda"
activities = [ProfilerActivity.CUDA]
sort_by_keyword = device + "_time_total"

model_cuda = model.to(device)
inputs_cuda = torch.randn(1024, 2).to(device)

with profile(activities=[ProfilerActivity.CUDA], record_shapes=True) as prof:
    with record_function("model_inference"):
        model_cuda(inputs_cuda)
        
print(prof.key_averages(group_by_input_shape=True).table(sort_by="cuda_time_total", row_limit=10))